# ICC Design Effect — Sample Size Adjuster

**Method:** Inflate planned sample size to account for within-cluster correlation (ICC) using the Design Effect (DEFF).

---

$$\text{DEFF} = 1 + (\bar{m} - 1) \times \text{ICC}$$

$$N_{\text{adjusted}} = N_{\text{planned}} \times \text{DEFF}$$

---

| Symbol | Meaning |
|--------|---------|
| $\text{ICC}$ | Intraclass Correlation Coefficient — within-cluster similarity |
| $\bar{m}$ | Average cluster size (= planned N / number of clusters) |
| $\text{DEFF}$ | Design Effect — inflation factor for effective sample size |
| $N_{\text{planned}}$ | Original planned sample size (from z-test calculator) |
| $N_{\text{adjusted}}$ | Adjusted sample size after accounting for clustering |

In [4]:
import json

config = json.load(open("../test_config.json"))
planned_n = config["computed"]["planned_n"]

# ==========================================
#  ICC PARAMETERS — enter your estimates
# ==========================================
icc_estimate = 0.01    # estimated ICC (typical web: 0.001–0.05)
num_clusters = 30      # number of clusters (days, campaigns, etc.)
# ==========================================

print(f"Planned N: {planned_n:,}  |  ICC estimate: {icc_estimate}  |  Clusters: {num_clusters}")

Planned N: 1,283  |  ICC estimate: 0.01  |  Clusters: 30


In [5]:
import numpy as np
import plotly.graph_objects as go

def calculate_icc_adjusted_n(planned_n, icc_estimate, num_clusters):
    m = planned_n / num_clusters
    deff = 1 + (m - 1) * icc_estimate
    adjusted_n = int(np.ceil(planned_n * deff))

    # --- Output ---
    print(f"--- ICC DESIGN EFFECT ADJUSTMENT ---")
    print(f"手法 : ICC デザイン効果によるサンプルサイズ補正")
    print(f"-" * 50)
    print(f"【パラメータ】")
    print(f"  計画サンプル数 (N)  : {planned_n:,}")
    print(f"  推定ICC             : {icc_estimate}")
    print(f"  クラスター数        : {num_clusters}")
    print(f"  平均クラスターサイズ : {m:.1f}")
    print(f"-" * 50)
    print(f"DEFF = 1 + (m̄ - 1) × ICC")
    print(f"     = 1 + ({m:.1f} - 1) × {icc_estimate}")
    print(f"     = {deff:.4f}")
    print(f"-" * 50)
    print(f"【補正結果】")
    print(f"  補正後サンプル数 : {adjusted_n:,}  (×{deff:.2f})")
    print(f"  増加分           : +{adjusted_n - planned_n:,}")
    print(f"-" * 50)

    # --- Sensitivity chart ---
    icc_range = np.linspace(0.001, 0.10, 500)
    adjusted_range = planned_n * (1 + (m - 1) * icc_range)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=icc_range, y=adjusted_range, mode='lines',
        name='Adjusted N', line=dict(color='#1f77b4', width=3)))
    fig.add_trace(go.Scatter(x=[icc_estimate], y=[adjusted_n], mode='markers',
        name=f'Your Estimate (ICC={icc_estimate})',
        marker=dict(color='red', size=12, symbol='diamond')))
    fig.add_hline(y=planned_n, line_dash="dot", line_color="gray",
        annotation_text=f"Planned N = {planned_n:,}", annotation_position="top left")
    fig.add_annotation(x=icc_estimate, y=adjusted_n,
        text=f"<b>{adjusted_n:,}</b><br>ICC={icc_estimate}",
        ax=40, ay=-40, font=dict(size=12, color="red"), arrowcolor="red")

    fig.update_layout(
        title=dict(text=f"<b>ICC Sensitivity — Adjusted Sample Size</b><br>"
            f"Planned N = {planned_n:,} | Clusters = {num_clusters} | m̄ = {m:.1f}",
            font=dict(size=20)),
        xaxis_title="ICC", yaxis_title="Adjusted N",
        xaxis=dict(showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    fig.show()

    return deff, adjusted_n

In [6]:
deff_icc, adjusted_n = calculate_icc_adjusted_n(
    planned_n=planned_n, icc_estimate=icc_estimate, num_clusters=num_clusters)

# Save to config
config["computed"]["deff_icc"] = round(deff_icc, 4)
config["computed"]["adjusted_n"] = adjusted_n
json.dump(config, open("../test_config.json", "w"), indent=2)

--- ICC DESIGN EFFECT ADJUSTMENT ---
手法 : ICC デザイン効果によるサンプルサイズ補正
--------------------------------------------------
【パラメータ】
  計画サンプル数 (N)  : 1,283
  推定ICC             : 0.01
  クラスター数        : 30
  平均クラスターサイズ : 42.8
--------------------------------------------------
DEFF = 1 + (m̄ - 1) × ICC
     = 1 + (42.8 - 1) × 0.01
     = 1.4177
--------------------------------------------------
【補正結果】
  補正後サンプル数 : 1,819  (×1.42)
  増加分           : +536
--------------------------------------------------
